### Finding vertex from initial point

In [1]:
import numpy as np
import pandas as pd

def findvertex(A, b, z, eps=1e-8, maxsteps=100, vb=True):
    m, n = A.shape
    z = np.array(z, dtype=float)
    tr = []  # tr means tight rows (kept linearly independent)

    def addtight():
        for i in range(m):
            if i not in tr and abs(np.dot(A[i], z) - b[i]) < eps:
                if np.linalg.matrix_rank(A[tr + [i]], tol=eps) > len(tr):
                    tr.append(i)

    addtight()
    step = 0
    while len(tr) < n and step < maxsteps:
        step += 1
        # directions that keep every currently tight row tight = null space of A[tr]
        if len(tr) > 0:
            _, _, Vt = np.linalg.svd(A[tr])
            N = Vt[len(tr):].T
        else:
            N = np.eye(n)

        moved = False
        for k in range(N.shape[1]):
            for sgn in (1, -1):
                d = sgn * N[:, k]
                sl = np.inf  # sl means step length
                ni = None
                for i in range(m):
                    if i in tr:
                        continue
                    Aid = np.dot(A[i], d)
                    if Aid > eps:
                        t = (b[i] - np.dot(A[i], z)) / Aid
                        if t < sl:
                            sl = t
                            ni = i
                if ni is not None:
                    moved = True
                    break
            if moved:
                break

        if not moved:
            if vb:
                print("No vertex reachable (feasible region contains a full line).")
            break

        z = z + sl * d
        addtight()
        if vb:
            print(f"Step {step}: Hit constraint {ni}, total tight = {len(tr)}")

    if vb:
        print("\nFinal tight rows:", tr)
    return np.round(z, 8), tr

##Read and display input

In [2]:
data = pd.read_csv("testcase_1.csv", header=None).to_numpy()

z = data[0, :-1] # Initial feasible point
c = data[1, :-1]   # Cost vector
A = data[2:, :-1] # Constraint matrix
b = data[2:, -1]   # Constraint vector

print("Initial feasible point (z):", z)
print("Cost vector (c):", c)
print("Constraint matrix (A):\n", A)
print("Constraint vector (b):", b)

Initial feasible point (z): [0 0]
Cost vector (c): [5 2]
Constraint matrix (A):
 [[ 0  1]
 [ 1  0]
 [-1 -1]]
Constraint vector (b): [1 1 1]


### Finding and displaying the tight rows

In [3]:
# ---- After reading z, A, b ----
print("\nMoving initial feasible point to a vertex...")
z, tr = findvertex(A, b, z)
print("Vertex found:", z)

m, n = A.shape
eps = 1e-8
if len(tr) < n:
    raise ValueError("No vertex found: A needs n linearly independent rows.")

# Use this z as starting vertex
A0 = A[tr].copy()
b0 = b[tr].copy()


Moving initial feasible point to a vertex...
Step 1: Hit constraint 1, total tight = 1
Step 2: Hit constraint 0, total tight = 2

Final tight rows: [1, 0]
Vertex found: [1. 1.]


### Starting iterative Simplex Algorithm

In [4]:
cp = z.copy() # cp means current point
dv = [cp] # dv is list of discovering vertices
costs = [np.dot(c, cp)]

it = 0 # it means iteration
unb = False # unb means unbounded
while True:
    it += 1
    print(f"\nIteration {it} :")

    # Finding direction matrix V = -A0^-1
    A0inv = np.linalg.inv(A0)
    V = -A0inv

    imp = False # imp means improved
    nextp = None # nextp means next point
    bc = np.dot(c, cp) # bc means best cost

    # Trying each direction (Bland's rule: lowest constraint index first, avoids cycling)
    for i in sorted(range(n), key=lambda k: tr[k]):
        v = V[:, i]
        chc = np.dot(c, v) # chc is change in objective function cost per unit step in direction v

        if chc <= eps:
            continue  # no improvement

        # Finding step length sl (ratio test over ALL rows, zero-length steps allowed)
        sl = np.inf # sl is max step length we can move along chosen direction v before hitting a new constraint
        tir = None # tir is index of row that becomes tight after moving in direction v by step sl
        for j in range(m):
            Ajv = np.dot(A[j], v)
            if Ajv > eps:
                tj = max((b[j] - np.dot(A[j], cp)) / Ajv, 0)
                if tj < sl - eps:
                    sl = tj
                    tir = j

        if tir is None:
            unb = True # improving direction with no blocking constraint
            break

        pp = cp + sl * v # pp means potential point
        pc = np.dot(c, pp) # pc means potential cost
        imp = True
        bc = pc
        nextp = pp
        rir = i # rir is the row index in the current tight set A0, which is to be replaced
        ni = tir # ni is the index of new constraint that became tight after moving to new vertex
        break

    if unb:
        print("Unbounded direction found. Problem is unbounded.")
        break

    if not imp:
        print("No improving direction found. Optimum reached.")
        break

    # Updating current point
    cp = nextp
    dv.append(cp)
    costs.append(bc)

    print(f"New vertex: {cp}")
    print(f"Objective value: {bc:.4f}")

    # Updating A0 (replacing one row)
    A0[rir] = A[ni]
    b0[rir] = b[ni]
    tr[rir] = ni


Iteration 1 :
No improving direction found. Optimum reached.


##Printing Result

In [5]:
# ---------- STEP 4: Print final results ----------
print("\nSequence of vertices visited:")
for i, v in enumerate(dv):
    print(f"Vertex {i}: {v}  ,  Cost = {costs[i]:.4f}")

if unb:
    print("\nProblem is unbounded (objective = inf)")
else:
    print("\nOptimal vertex:", dv[-1])
    print("Optimal cost:", costs[-1])


Sequence of vertices visited:
Vertex 0: [1. 1.]  ,  Cost = 7.0000

Optimal vertex: [1. 1.]
Optimal cost: 7.0
